# KAP Disclosure List API

**Türkiye Kamuyu Aydınlatma Platformu** — payı işlem gören şirketlerin özel durum, finansal rapor, duyuru ve diğer bildirimlerinin resmi listesi.

Tarih aralığı + bildirim tipi + üye tipi kombinasyonu ile filtre; tek şirket için `mkkMemberOid` ile daraltma.

## Endpoint

```
POST https://kap.org.tr/tr/api/disclosure/list/main
```

**Header:** `Content-Type: application/json` · **Auth:** yok · **Yanıt:** `Array<{ disclosureBasic, disclosureDetail }>` · en yeni üstte

## İstek (Request) Alanları

| Alan | Açıklama |
|------|----------|
| `fromDate` | Başlangıç tarihi · `GG.AA.YYYY` |
| `toDate` | Bitiş tarihi · `GG.AA.YYYY` |
| `disclosureTypes` | Bildirim tipleri dizisi |
| `memberTypes` | Üye tipleri dizisi |
| `mkkMemberOid` | Tek şirket filtresi · `null` ise tümü |

### `disclosureTypes` kodları

| Kod | Anlam |
|-----|-------|
| `ODA` | Özel Durum Açıklaması |
| `FR` | Finansal Rapor |
| `DUY` | Duyuru |
| `DG` | Diğer |
| `CA` | Corporate Action (Kâr payı, Genel Kurul, vb.) |

### `memberTypes` kodları

| Kod | Anlam |
|-----|-------|
| `PYS` | Payı işlem gören şirketler |
| `DDK` | Bağımsız denetim kuruluşları |
| `KVH` | Kolektif yatırım kuruluşları |
| `DG` | Diğer üyeler |
| `IGS` | İhraççı bilgi formu sahipleri |
| `YK` | Yatırım kuruluşları |

## Yanıt (Response)

Her bildirim `{ disclosureBasic, disclosureDetail }` çiftidir; en yeni en üstte.

### `disclosureBasic`

| Alan | Açıklama |
|------|----------|
| `disclosureIndex` | Benzersiz ID — detay linkinde kullanılır |
| `disclosureType` | Tip kodu (`ODA`, `FR`, `DUY`, `DG`, `CA`) |
| `companyTitle` | Şirket unvanı |
| `stockCode` | BIST kodu (boş olabilir) |
| `relatedStocks` | İlişkili hisseler · virgülle ayrık string / `null` |
| `title` | Bildirim başlığı |
| `summary` | Özet · `null` olabilir |
| `publishDate` | Yayım tarihi-saati · `GG.AA.YYYY HH:MM:SS` |
| `attachmentCount` | Eklenti sayısı |
| `period` / `year` / `donem` | Dönem / yıl bilgisi (FR'de dolu) |
| `isLate` / `isBlocked` | Gecikmeli / engelli |
| `hasMultiLanguageSupport` | `Y` / `N` |
| `mkkMemberOid` | Şirket OID — tek şirket filtresi için |

### `disclosureDetail`

| Alan | Açıklama |
|------|----------|
| `memberType` | Üye tipi |
| `auditType` | Denetim tipi |
| `opinion` / `opinionType` | Denetçi görüşü |
| `ftNiteligi` | Finansal tablo niteliği |
| `fundOid` | Fon kimliği (kolektif yatırım) |
| `relatedDisclosureIndex` | İlişkili bildirim |
| `oldKap` | Eski KAP bildirimi mi |

**Detay linki:** `https://kap.org.tr/tr/Bildirim/{disclosureIndex}`

## 1. Ortak Ayarlar

In [ ]:
import requests
from datetime import datetime, timedelta

URL = "https://kap.org.tr/tr/api/disclosure/list/main"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
}

DISCLOSURE_TYPES_ALL = ["ODA", "FR", "DUY", "DG", "CA"]
MEMBER_TYPES_ALL = ["PYS", "DDK", "KVH", "DG", "IGS", "YK"]


def trf(d):
    """date / datetime → KAP'ın beklediği GG.AA.YYYY string'i."""
    return d.strftime("%d.%m.%Y")

## 2. Tarama Fonksiyonu

Tarih aralığı + opsiyonel tip / üye / şirket filtreleri ile bildirimleri çeker; sonuçları okunur tabloya yazar.

In [ ]:
def bildirimler(from_date, to_date, disclosure_types=None, member_types=None, mkk_member_oid=None):
    body = {
        "fromDate": trf(from_date),
        "toDate": trf(to_date),
        "disclosureTypes": disclosure_types or DISCLOSURE_TYPES_ALL,
        "memberTypes": member_types or MEMBER_TYPES_ALL,
        "mkkMemberOid": mkk_member_oid,
    }
    r = requests.post(URL, headers=HEADERS, json=body)
    r.raise_for_status()
    return r.json()


def yazdir(rows, baslik, limit=20):
    print(f"{baslik} · {len(rows)} bildirim · ilk {min(limit, len(rows))}\n")
    print(f"{'Saat':<17} {'Kod':<10} {'Tip':<5} {'Geç':<4}  Başlık")
    print("-" * 100)
    for r in rows[:limit]:
        b = r["disclosureBasic"]
        saat = (b.get("publishDate") or "")[:16]
        kod = (b.get("stockCode") or "-")[:10]
        tip = (b.get("disclosureType") or "-")[:5]
        gec = "evet" if b.get("isLate") else "-"
        title = (b.get("title") or "").strip().replace("\n", " ")[:55]
        print(f"{saat:<17} {kod:<10} {tip:<5} {gec:<4}  {title}")

## 3. Bugün · Tüm Bildirimler

Son 24 saatlik pencere · tip filtresi yok (hepsi).

In [3]:
bugun = datetime.now().date()
rows = bildirimler(bugun - timedelta(days=1), bugun)
yazdir(rows, f"KAP · {trf(bugun - timedelta(days=1))}–{trf(bugun)}")

KAP · 30.04.2026–01.05.2026 · 946 bildirim · ilk 20

Saat              Kod        Tip   Geç   Başlık
----------------------------------------------------------------------------------------------------
01.05.2026 11:05  FLAP       CA    -     Genel Kurul İşlemlerine İlişkin Bildirim
01.05.2026 02:41  VKFYO      FR    evet  Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 02:41  VKFYO      FR    evet  Faaliyet Raporu (Konsolide Olmayan)
01.05.2026 02:41  VKFYO      FR    evet  Finansal Rapor
01.05.2026 01:10  FBY        FR    evet  Finansal Rapor
01.05.2026 01:06  FBY        FR    evet  Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 00:37  OZATD      DG    -     Pay  Satış Bilgi Formu
01.05.2026 00:22  NVG        FR    evet  Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 00:18  SDTTR      FR    -     Sorumluluk Beyanı (Konsolide)
01.05.2026 00:18  NVG        FR    evet  Faaliyet Raporu (Konsolide Olmayan)
01.05.2026 00:18  SDTTR      FR    -     Faaliyet Raporu (Konsolide)
01.05.202

## 4. Sadece Finansal Raporlar (FR)

`disclosureTypes=["FR"]` ile finansal raporlar; BIST kodu olanlar dönem & yıl bilgisiyle.

In [4]:
rows = bildirimler(bugun - timedelta(days=7), bugun, disclosure_types=["FR"])
fr = [r for r in rows if r["disclosureBasic"].get("stockCode")]

print(f"FR · son 7 gün · BIST kodu olanlar · {len(fr)} kayıt · ilk 15\n")
print(f"{'Saat':<17} {'Kod':<10} {'Dönem':<6} {'Yıl':<5}  Başlık")
print("-" * 95)
for r in fr[:15]:
    b = r["disclosureBasic"]
    saat = (b.get("publishDate") or "")[:16]
    kod = (b.get("stockCode") or "-")[:10]
    donem = str(b.get("period") or "-")[:6]
    yil = str(b.get("year") or "-")[:5]
    title = (b.get("title") or "").strip()[:50]
    print(f"{saat:<17} {kod:<10} {donem:<6} {yil:<5}  {title}")

FR · son 7 gün · BIST kodu olanlar · 941 kayıt · ilk 15

Saat              Kod        Dönem  Yıl    Başlık
-----------------------------------------------------------------------------------------------
01.05.2026 02:41  VKFYO      3AB    2026   Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 02:41  VKFYO      3AB    2026   Faaliyet Raporu (Konsolide Olmayan)
01.05.2026 02:41  VKFYO      3AB    2026   Finansal Rapor
01.05.2026 01:10  FBY        3AB    2026   Finansal Rapor
01.05.2026 01:06  FBY        3AB    2026   Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 00:22  NVG        3AB    2026   Sorumluluk Beyanı (Konsolide Olmayan)
01.05.2026 00:18  SDTTR      3AB    2026   Sorumluluk Beyanı (Konsolide)
01.05.2026 00:18  NVG        3AB    2026   Faaliyet Raporu (Konsolide Olmayan)
01.05.2026 00:18  SDTTR      3AB    2026   Faaliyet Raporu (Konsolide)
01.05.2026 00:17  SDTTR      3AB    2026   Finansal Rapor
01.05.2026 00:09  NVG        3AB    2026   Finansal Rapor
30.04.2026 23:49  GA

## 5. Belirli Bir Şirket · `mkkMemberOid`

Şirket OID'si ile son 60 günün tüm bildirimleri. OID, şirketin geçmiş bildirimlerinin `mkkMemberOid` alanından okunabilir.

Örnek: **THYAO** · `4028e4a140f2ed720140f376bebb01a7`.

In [5]:
THY_OID = "4028e4a140f2ed720140f376bebb01a7"

rows = bildirimler(bugun - timedelta(days=60), bugun, mkk_member_oid=THY_OID)

print(f"THYAO · son 60 gün · {len(rows)} bildirim\n")
print(f"{'Saat':<17} {'Tip':<5} {'idx':>9}  Başlık")
print("-" * 95)
for r in rows[:15]:
    b = r["disclosureBasic"]
    saat = (b.get("publishDate") or "")[:16]
    tip = (b.get("disclosureType") or "-")[:5]
    idx = b.get("disclosureIndex")
    title = (b.get("title") or "").strip()[:55]
    print(f"{saat:<17} {tip:<5} {idx:>9}  {title}")

if rows:
    idx = rows[0]["disclosureBasic"]["disclosureIndex"]
    print(f"\nDetay: https://kap.org.tr/tr/Bildirim/{idx}")

THYAO · son 60 gün · 29 bildirim

Saat              Tip         idx  Başlık
-----------------------------------------------------------------------------------------------
29.04.2026 18:21  ODA     1598902  Özel Durum Açıklaması (Genel)
29.04.2026 18:21  FR      1598901  Sorumluluk Beyanı (Konsolide)
29.04.2026 18:21  FR      1598900  Faaliyet Raporu (Konsolide)
29.04.2026 18:20  FR      1598897  Finansal Rapor
14.04.2026 11:55  CA      1592906  Genel Kurul İşlemlerine İlişkin Bildirim
09.04.2026 21:56  DG      1590424  Şirket Genel Bilgi Formu
09.04.2026 19:17  ODA     1590373  Özel Durum Açıklaması (Genel)
09.04.2026 19:16  ODA     1590371  Özel Durum Açıklaması (Genel)
09.04.2026 19:16  ODA     1590370  Özel Durum Açıklaması (Genel)
09.04.2026 19:11  CA      1590365  Kar Payı Dağıtım İşlemlerine İlişkin Bildirim
09.04.2026 18:59  CA      1590360  Genel Kurul İşlemlerine İlişkin Bildirim
09.04.2026 08:42  ODA     1589738  Özel Durum Açıklaması (Genel)
01.04.2026 17:56  ODA     158237

## 6. curl Eşdeğeri

Aynı isteği shell'den; `jq` ile en yeni 10 bildirimi tabloya çevir.

In [4]:
%%bash
curl -s -X POST "https://kap.org.tr/tr/api/disclosure/list/main" \
  -H "Content-Type: application/json" \
  -d '{
    "fromDate": "01.04.2026",
    "toDate":   "30.04.2026",
    "disclosureTypes": ["ODA","FR","DUY","DG","CA"],
    "memberTypes":     ["PYS","DDK","KVH","DG","IGS","YK"],
    "mkkMemberOid": null
  }' \
  | jq -r '.[0:10]
      | (["SAAT","KOD","TIP","BASLIK"]),
        (["----","---","---","------"]),
        (.[] | [
            .disclosureBasic.publishDate[0:16],
            (.disclosureBasic.stockCode // "-"),
            .disclosureBasic.disclosureType,
            (.disclosureBasic.title | gsub("\n"; " ") | .[0:55])
          ])
      | @tsv' \
  | column -t -s $'\t' 

SAAT              KOD    TIP  BASLIK
----              ---    ---  ------
30.04.2026 23:56  OZATD  DG   Şirket Genel Bilgi Formu
30.04.2026 23:55  YESIL  DG   Şirket Genel Bilgi Formu
30.04.2026 23:49  GATEG  FR   Sorumluluk Beyanı (Konsolide Olmayan)
30.04.2026 23:47  GATEG  FR   Faaliyet Raporu (Konsolide Olmayan)
30.04.2026 23:43  BURCE  ODA  Finansal Duran Varlık Satışı
30.04.2026 23:43  ZERGY  FR   Sorumluluk Beyanı (Konsolide Olmayan)
30.04.2026 23:42  ZERGY  FR   Faaliyet Raporu (Konsolide Olmayan)
30.04.2026 23:42  ZERGY  FR   Finansal Rapor
30.04.2026 23:35  GATEG  FR   Finansal Rapor
30.04.2026 23:34  OZATD  ODA  Özel Durum Açıklaması (Genel)


## Notlar

- **Tarih formatı:** `fromDate` / `toDate` kesinlikle `GG.AA.YYYY`. ISO 8601 veya `YYYY-MM-DD` kabul edilmez.
- **Liste filtreleri:** `disclosureTypes` ve `memberTypes` boş `[]` gönderilirse hiçbir kayıt dönmez — istemediğin tipi listeden çıkar, listeyi boş bırakma.
- **`stockCode` boş olabilir:** Yatırım fonları, bağımsız denetim kuruluşları gibi BIST'te işlem görmeyen üyelerde `stockCode` `null`. Hisse bazlı filtreleme yapacaksan `null` kontrolü gerek.
- **`relatedStocks`:** Bir bildirim birden fazla hisseye ait olabilir; birincil `stockCode` dışında `relatedStocks` (virgülle ayrık string) kontrol edilmeli.
- **Tek şirket filtresi:** `mkkMemberOid` şirket sayfasının URL'inden veya geçmiş bildirimlerin `mkkMemberOid` alanından okunur — BIST kodundan otomatik çıkmaz.
- **Çift kayıt yok ama benzer başlık:** Aynı bildirim için "Finansal Rapor", "Sorumluluk Beyanı", "Faaliyet Raporu" ayrı `disclosureIndex`'lerle gelir — finansal rapor başına 2-4 satır beklenir.
- **`disclosureDetail` çoğu durumda boş:** Denetçi görüşü / `ftNiteligi` / `auditType` alanları sadece konsolide finansal rapor + denetim raporu kayıtlarında dolar; gündelik özel durumlarda hepsi `null`.
- **Detay linki:** `https://kap.org.tr/tr/Bildirim/{disclosureIndex}` — eklenti varsa orada listelenir, ayrı bir endpoint yok.
- **Auth yok, rate-limit belgelenmemiş:** Polling'i makul tut; CDN cache'i nedeniyle 1 dk altı tazelemeler yeni veri getirmeyebilir.